In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sentence_transformer,
    load_triplets,
)


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Block 2: Evaluate PER-EPOCH MNRL checkpoints (LoRA adapters -> merge -> eval)

import shutil
from pathlib import Path
from sentence_transformers import SentenceTransformer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"

DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_dbpedia.json")
CKPT_ROOT = Path(PROJECT_ROOT) / "../baseline/checkpoints/qwen_dbpedia_mnrl/checkpoints"
TMP_ROOT  = Path(PROJECT_ROOT) / "../baseline/checkpoints/qwen_dbpedia_mnrl/tmp_merged_eval"
TMP_ROOT.mkdir(parents=True, exist_ok=True)

TOPKS = [3, 5, 10, 20]
BATCH = 32
STEPS_PER_EPOCH = 3934   # from your MNRL run log
EPOCHS = [1, 2]          # change if you trained more

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

for epoch in EPOCHS:
    step = epoch * STEPS_PER_EPOCH
    adapter_dir = (CKPT_ROOT / f"checkpoint-{step}").resolve()
    merged_dir  = (TMP_ROOT / f"epoch_{epoch}_merged").resolve()

    print(f"\n=== Evaluating MNRL epoch {epoch} ({adapter_dir}) ===")

    # 1) load base ST
    model = SentenceTransformer(BASE_MODEL)

    # 2) attach LoRA adapter
    word_module = model[0]
    base_hf = word_module.auto_model
    peft_model = PeftModel.from_pretrained(base_hf, str(adapter_dir))

    # 3) merge adapter into base
    word_module.auto_model = peft_model.merge_and_unload()

    # 4) save temporary merged folder
    if merged_dir.exists():
        shutil.rmtree(merged_dir)
    model.save(str(merged_dir))

    # 5) evaluate
    evaluate_sentence_transformer(
        model_name=str(merged_dir),
        display_name=f"Fine-tuned (MNRL): Qwen DBpedia (epoch {epoch})",
        eval_items=eval_items,
        global_corpus=global_corpus,
        relevance_scores_per_item=relevance_scores_per_item,
        topks=TOPKS,
        batch_size=BATCH,
    )


=== Evaluating MNRL epoch 1 (/local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/baseline/checkpoints/qwen_dbpedia_mnrl/checkpoints/checkpoint-3934) ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6086.24it/s]



Fine-tuned (MNRL): Qwen DBpedia (epoch 1)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1098  0.0374  0.0507  0.1419  0.2136    0.1098  0.1044
@5     0.1122  0.0625  0.0716  0.1606  0.2938    0.1122  0.1127
@10    0.1202  0.1266  0.1085  0.1825  0.4599    0.1202  0.1344
@20    0.1245  0.2480  0.1484  0.1942  0.6261    0.1245  0.1823

=== Evaluating MNRL epoch 2 (/local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/baseline/checkpoints/qwen_dbpedia_mnrl/checkpoints/checkpoint-7868) ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4789.78it/s]



Fine-tuned (MNRL): Qwen DBpedia (epoch 2)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1187  0.0396  0.0534  0.1513  0.2344    0.1187  0.1121
@5     0.1199  0.0636  0.0742  0.1644  0.2908    0.1199  0.1195
@10    0.1249  0.1292  0.1126  0.1858  0.4570    0.1249  0.1386
@20    0.1325  0.2622  0.1574  0.1981  0.6291    0.1325  0.1912
